# Final PCA: covariates for phenotype residualization

Round 1/2/2b's PCAs all exist to *classify ancestry* -- fit on 1000G-comparable variants (`agreeing_snps.ids`, round 1's ID+REF+ALT-verified, HM3-restricted set) so AoU samples can be projected into a space where 1000G population labels are meaningful, and (round 2/2b) refit tighter for a purity threshold. That HM3 restriction is necessary for classification, but it's a real cost once the ancestry decision is already made: HM3 is ascertainment-biased toward common EUR variants (chosen for the original HapMap project), which distorts PC resolution for non-EUR cohorts -- most concretely the `afr` sample set -- and its ~1.3M-variant density is much lower than what's actually available.

This notebook is a *separate, later* step: once a sample set's ancestry-filtered cohort is fixed (round 2b's keep-list), it fits a fresh PCA directly on `genome_wide_qc_thinning_merge.ipynb`'s already-built panel -- QC'd and LD-pruned from the **full ACAF catalog**, not HM3-restricted, ~1M variants, and already computed for GRM construction, so no new pruning work is needed here. No 1000G projection this time -- that's the point: this PCA's only job is producing good covariates for `residualize_phenotypes.ipynb`, not classifying anyone, so there's no need to keep it on a reference-comparable variant set. `--pca approx` (Galinsky et al. 2016, "fastPCA"), same as `reverse_pca_aou.ipynb`, since this runs on the full round-2b-passing cohort, not a subsample.

Depends on `genome_wide_qc_thinning_merge.ipynb` (`03_grm_shards`) having already been run for this `SAMPLE_SET` -- this notebook only reads that panel, it doesn't build it.

## Compute resource

Same panel `grm_shard_timing.ipynb`/`grm_shard_run.ipynb` size for -- a real ~1M-variant, full-cohort genotype panel, CPU + I/O bound. Size similarly to those notebooks (16+ vCPU, enough RAM for `--memory` to cap plink2's own default reservation) rather than Workbench 2.0's small default. Much lighter than GRM shard construction itself (this is one PCA fit, not `N_SHARDS` `--make-grm-bin` calls), but still not a task for the default 2 CPU / 13 GB environment.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

N_THREADS = os.cpu_count()

## Inputs

`MERGED_PREFIX` points at `genome_wide_qc_thinning_merge.ipynb`'s pgen output for this `CDR_VERSION`/`SAMPLE_SET` (`genome_wide_round2b_thinned_{CDR_VERSION}_{SAMPLE_SET}` -- the pgen form, not the `_bed` PLINK1 export that notebook also writes for the GRM step; plink2 reads pgen natively via `--pfile`, no need for the bed conversion here). Copied to local scratch first, same convention as everywhere else in this pipeline -- plink2 reading a multi-GB panel repeatedly over the gcsfuse-mounted bucket is much slower than local disk.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)

# Must match genome_wide_qc_thinning_merge.ipynb's CDR_VERSION/SAMPLE_SET -- this
# notebook only reads that notebook's merged panel, it doesn't build one.
CDR_VERSION = "v9"
SAMPLE_SET = "eur"   # <-- change this and rerun for each of the 5 sample sets

BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/03_grm_shards/{SAMPLE_SET}"
FINAL_PCA_BUCKET_DIR = f"{BUCKET_DIR}/final_pca"
os.makedirs(FINAL_PCA_BUCKET_DIR, exist_ok=True)

MERGED_NAME = f"genome_wide_round2b_thinned_{CDR_VERSION}_{SAMPLE_SET}"   # pgen form, from genome_wide_qc_thinning_merge.ipynb

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")   # same scratch dir genome_wide_qc_thinning_merge.ipynb/grm_shard_timing.ipynb use -- panel may already be there
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{BUCKET_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged panel: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's "
        f"merge section for SAMPLE_SET={SAMPLE_SET!r} first"
    )
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

N_PCS = 20   # matches every other PC output in this pipeline (round 2b's PC_COVARIATE_PATH, etc.)
FINAL_PCA_PREFIX = os.path.join(LOCAL_WORK_DIR, f"final_pca_{CDR_VERSION}_{SAMPLE_SET}")

print(MERGED_PREFIX)
print(FINAL_PCA_BUCKET_DIR)

## Fit the PCA

`--pca approx`: fits on the full round-2b-passing cohort directly (no subsampling), same reasoning as `reverse_pca_aou.ipynb` -- plink2's exact PCA algorithm doesn't scale to a cohort this size, so this uses the randomized/Blanczos algorithm (Galinsky et al. 2016, "fastPCA") instead. No `--keep` needed -- `genome_wide_qc_thinning_merge.ipynb`'s panel is already restricted to round 2b's keep-list (that's `ROUND2B_KEEP_PATH` in that notebook), and no HWE/LD-pruning here either -- both already applied when that panel was built (`--maf 0.01 --hwe 1e-6 ... --geno 0.05 --thin ...`), so this cell is just the PCA fit itself, not a QC pass.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$FINAL_PCA_PREFIX" "$N_THREADS" "$N_PCS"
set -e
MERGED_PREFIX=$1
FINAL_PCA_PREFIX=$2
THREADS=$3
NPCS=$4

plink2 \
  --pfile "$MERGED_PREFIX" \
  --nonfounders \
  --freq counts \
  --pca approx "$NPCS" \
  --threads "$THREADS" \
  --out "$FINAL_PCA_PREFIX"

ls -lh "${FINAL_PCA_PREFIX}".*

## Scree plot

Quick sanity check -- % variance explained per PC. A PCA fit on a much denser, unbiased-by-HM3-ascertainment panel should still show the expected steep drop-off after the first few structure-carrying PCs; a flat/noisy scree here would be a sign something's off in the input panel, not necessarily in this fit.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eigenval = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Final PCA [{SAMPLE_SET}] scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_pca_scree_{SAMPLE_SET}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Write PC covariates for residualize_phenotypes.ipynb

Same `IID PC1 ... PC20` format `residualize_phenotypes.ipynb`'s `PC_PATH` / `pull_covariates()` expects (and the same format `residualize_phenotypes_round2.ipynb`'s "Build round 2 PC covariates" cell produces) -- plink2's direct `.eigenvec` output already has one row per sample, just needs the `#FID`/`FID` column dropped and the ID column normalized to `IID`.

In [ ]:
direct = pd.read_csv(f"{FINAL_PCA_PREFIX}.eigenvec", sep=r"\s+")

id_col = "#IID" if "#IID" in direct.columns else "IID"
pc_cols = [c for c in direct.columns if c.startswith("PC")]
assert len(pc_cols) == N_PCS, f"expected {N_PCS} PC columns, found {len(pc_cols)}: {pc_cols}"

covariate_table = direct[[id_col] + pc_cols].rename(columns={id_col: "IID"})
covariate_table["IID"] = covariate_table["IID"].astype(str)

PC_COVARIATE_PATH = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_pca_pc_covariates_{SAMPLE_SET}.txt")
covariate_table.to_csv(PC_COVARIATE_PATH, sep="\t", index=False)
print(f"Wrote {len(covariate_table)} samples' PC1-PC{N_PCS} -> {PC_COVARIATE_PATH}")

## Next steps

Not wired into a residualization notebook yet -- `residualize_phenotypes.ipynb`/`residualize_phenotypes_round2.ipynb` still point `PC_PATH` at round 2b's/round 2's own covariate output. To use this notebook's PCs instead, either point one of those notebooks' `PC_PATH` at `PC_COVARIATE_PATH` above (keeping `KEEP_LIST_PATH` as-is, since the sample set itself is unchanged -- only the PC covariates differ), or add a third `residualize_phenotypes_final_pca.ipynb` twin, same pattern as the round-2 one, once this fit has been checked out for real.